# Model 3: Pre-Transfer Qualities + Team Styles — Midfielders## What changes vs Model 2?**Model 2** solo miraba al jugador: *"¿cómo jugaba antes?"* → predice cómo juega después.**Model 3** agrega contexto táctico: *"¿cómo jugaba antes?"* + *"¿de qué tipo de equipo viene y a qué tipo de equipo va?"*---## Ecuación$$\text{to}_{Q_i} = \alpha + \underbrace{\sum_{j=1}^{17} \beta_j \cdot \text{from}_{Q_j}}_{\text{qualities del jugador antes}} + \underbrace{\sum_{k=1}^{7} \gamma_k \cdot \text{team\_from}_{k}}_{\text{estilo equipo origen}} + \underbrace{\sum_{k=1}^{7} \delta_k \cdot \text{team\_to}_{k}}_{\text{estilo equipo destino}}$$En español:> **Quality después = f( cómo jugaba el jugador + cómo jugaba su equipo anterior + cómo juega su equipo nuevo )**## Features (31 en total)| Grupo | Cantidad | Ejemplo ||-------|----------|---------|| Player qualities pre-transfer | 17 | from_Pressing, from_Dribbling, ... || Team origin (projected) | 7 | from_q_proj_DEFENCE, from_q_proj_ATTACK, ... || Team destination (current) | 7 | to_q_DEFENCE, to_q_ATTACK, ... |**Nota:** Las team qualities del equipo origen son *proyectadas* (estimación de cómo jugará la siguiente temporada), porque al momento del transfer todavía no termina la nueva temporada.---

In [ ]:
import pandas as pdimport numpy as npimport statsmodels.api as smfrom sklearn.metrics import r2_score, mean_absolute_error, mean_squared_errorimport plotly.graph_objects as goimport plotly.express as pxfrom plotly.subplots import make_subplotsimport warningswarnings.filterwarnings("ignore")pd.set_option("display.max_columns", 40)pd.set_option("display.float_format", "{:.4f}".format)DATA = "../../../../thesis_data/processed_data/thesis_model_dataset/active/within_league_transfers_v5.parquet"df = pd.read_parquet(DATA)mf = df[df["from_position"] == "Midfielder"].copy()print(f"Midfielders: {len(mf):,} rows")

In [ ]:
QUALITIES = [    "Active defence", "Aerial threat", "Box threat", "Composure",    "Defensive heading", "Dribbling", "Effectiveness", "Finishing",    "Hold-up play", "Intelligent defence", "Involvement",    "Passing quality", "Pressing", "Progression",    "Providing teammates", "Run quality", "Winning duels",]TEAM_Q = ["DEFENCE", "DEFENSIVE_TRANSITION", "ATTACKING_TRANSITION",          "ATTACK", "PENETRATION", "CHANCE_CREATION", "OUTCOME"]TEAM_Q_LABELS = {    "DEFENCE": "Defence", "DEFENSIVE_TRANSITION": "Def. Transition",    "ATTACKING_TRANSITION": "Att. Transition", "ATTACK": "Attack",    "PENETRATION": "Penetration", "CHANCE_CREATION": "Chance Creation",    "OUTCOME": "Outcome",}from_pq  = [f"from_{q}" for q in QUALITIES]to_pq    = [f"to_{q}" for q in QUALITIES]from_tq  = [f"from_q_proj_{q}" for q in TEAM_Q]to_tq    = [f"to_q_{q}" for q in TEAM_Q]all_features = from_pq + from_tq + to_tq  # 17 + 7 + 7 = 31mf_clean = mf[all_features + to_pq + ["from_season"]].dropna()train = mf_clean[mf_clean["from_season"] <= 2023]test  = mf_clean[mf_clean["from_season"] == 2024]print(f"Clean: {len(mf_clean):,} | Train: {len(train):,} | Test: {len(test):,}")print(f"Features: {len(all_features)} (17 player + 7 team origin + 7 team dest)")

## Visualización: Team Qualities de un TransferPara un transfer de ejemplo, ¿cómo se comparan los estilos tácticos del equipo origen vs destino?

In [ ]:
ex = mf_clean.iloc[100]labels = [TEAM_Q_LABELS[q] for q in TEAM_Q]from_vals = [ex[f"from_q_proj_{q}"] for q in TEAM_Q]to_vals   = [ex[f"to_q_{q}"] for q in TEAM_Q]fig = go.Figure()fig.add_trace(go.Bar(name="Equipo origen (proj)", x=labels, y=from_vals, marker_color="#636EFA"))fig.add_trace(go.Bar(name="Equipo destino (actual)", x=labels, y=to_vals, marker_color="#EF553B"))fig.update_layout(title="Ejemplo: Estilo táctico — equipo origen vs destino",    yaxis_title="Quality score (z-score)", barmode="group",    height=400, template="plotly_white")fig.show()

---## Resultados31 features → predice cada `to_Qi` por separado (17 modelos OLS).

In [ ]:
results = []models = {}for q in QUALITIES:    X_train = sm.add_constant(train[all_features])    X_test  = sm.add_constant(test[all_features])    y_train = train[f"to_{q}"]    y_test  = test[f"to_{q}"]    model = sm.OLS(y_train, X_train).fit()    y_pred = model.predict(X_test)    results.append({        "Quality": q,        "R²_train": model.rsquared,        "R²_adj": model.rsquared_adj,        "R²_test": r2_score(y_test, y_pred),        "MAE_test": mean_absolute_error(y_test, y_pred),        "F_pvalue": model.f_pvalue,    })    models[q] = modelres = pd.DataFrame(results)print(res.to_string(index=False))print(f"\nMean R²_train: {res['R²_train'].mean():.4f}")print(f"Mean R²_adj:   {res['R²_adj'].mean():.4f}")print(f"Mean R²_test:  {res['R²_test'].mean():.4f}")print(f"Mean MAE_test: {res['MAE_test'].mean():.4f}")

### Comparación: Model 1 vs 2 vs 3¿Cuánto mejora agregar cross-qualities (M2) y luego team styles (M3)?

In [ ]:
# Recompute M1 and M2 for comparisonm1_r2, m2_r2 = [], []for q in QUALITIES:    # M1: self only    X1_tr = sm.add_constant(train[[f"from_{q}"]])    X1_te = sm.add_constant(test[[f"from_{q}"]])    m1 = sm.OLS(train[f"to_{q}"], X1_tr).fit()    m1_r2.append(r2_score(test[f"to_{q}"], m1.predict(X1_te)))    # M2: all 17 pre-Q    X2_tr = sm.add_constant(train[from_pq])    X2_te = sm.add_constant(test[from_pq])    m2 = sm.OLS(train[f"to_{q}"], X2_tr).fit()    m2_r2.append(r2_score(test[f"to_{q}"], m2.predict(X2_te)))comp = pd.DataFrame({    "Quality": QUALITIES,    "M1_self": m1_r2,    "M2_all_pq": m2_r2,    "M3_pq+tq": res["R²_test"].values,})comp["Δ M2→M3"] = comp["M3_pq+tq"] - comp["M2_all_pq"]print(comp.to_string(index=False))print(f"\nMean R²_test:  M1={comp['M1_self'].mean():.4f}  M2={comp['M2_all_pq'].mean():.4f}  M3={comp['M3_pq+tq'].mean():.4f}")print(f"Mean Δ M2→M3: {comp['Δ M2→M3'].mean():.4f}")print(f"Qualities improved M2→M3: {(comp['Δ M2→M3'] > 0).sum()} / {len(comp)}")

In [ ]:
fig = go.Figure()fig.add_trace(go.Bar(name="M1: self only", x=comp["Quality"], y=comp["M1_self"], marker_color="#636EFA"))fig.add_trace(go.Bar(name="M2: all 17 pre-Q", x=comp["Quality"], y=comp["M2_all_pq"], marker_color="#EF553B"))fig.add_trace(go.Bar(name="M3: + team styles", x=comp["Quality"], y=comp["M3_pq+tq"], marker_color="#00CC96"))fig.update_layout(title="R²_test — Model 1 vs 2 vs 3 (Midfielders)",    yaxis_title="R²_test", barmode="group", height=450, template="plotly_white",    xaxis_tickangle=-45)fig.show()

### Full OLS Summary — Best and Worst

In [ ]:
best_q = res.loc[res["R²_test"].idxmax(), "Quality"]worst_q = res.loc[res["R²_test"].idxmin(), "Quality"]print(f"BEST: {best_q} (R²_test = {res.loc[res['R²_test'].idxmax(), 'R²_test']:.4f})")print(models[best_q].summary())print(f"\n{'='*70}")print(f"WORST: {worst_q} (R²_test = {res.loc[res['R²_test'].idxmin(), 'R²_test']:.4f})")print(models[worst_q].summary())

### ¿Qué team qualities importan más?Coeficientes promedio (absolutos) de las 14 team features across all 17 models. Muestra qué dimensiones tácticas del equipo más afectan las qualities del jugador.

In [ ]:
# Average absolute coefficient for each team feature across all 17 modelsteam_features = from_tq + to_tqteam_coefs = pd.DataFrame(index=team_features, columns=["mean_abs_coef", "mean_coef"], dtype=float)for feat in team_features:    coefs = [models[q].params[feat] for q in QUALITIES]    team_coefs.loc[feat, "mean_abs_coef"] = np.mean(np.abs(coefs))    team_coefs.loc[feat, "mean_coef"] = np.mean(coefs)team_coefs = team_coefs.sort_values("mean_abs_coef", ascending=True)# Clean names for displayclean_names = [n.replace("from_q_proj_", "FROM: ").replace("to_q_", "TO: ") for n in team_coefs.index]fig = go.Figure(go.Bar(x=team_coefs["mean_abs_coef"].values, y=clean_names,    orientation="h", marker_color=["#636EFA" if "FROM" in n else "#EF553B" for n in clean_names]))fig.update_layout(title="Importancia promedio de cada team quality (|coef| promedio across 17 models)",    xaxis_title="Mean |coefficient|", height=450, template="plotly_white")fig.show()

---## TakeawayAgregar el **estilo táctico de los equipos** (14 features extra) mejora la predicción vs solo usar qualities del jugador.La mejora varía por quality — algunas como **Involvement** y **Composure** se benefician mucho del contexto táctico (tiene sentido: un equipo que juega más posesión genera más Involvement), mientras que otras como **Defensive heading** apenas cambian (es más una habilidad física individual).**Siguiente modelo:** En vez de predecir `to_Qi` directamente, predecir el **cambio** (`delta_Qi = to_Qi - from_Qi`) usando las qualities pre-transfer y los **deltas** de team qualities (`to_team - from_team`).